# 03 · Clean: missing values & outliers (Issue #3)

Turns Skylar's EDA findings (notebooks 01 and 02) into a reusable cleaning step.

```
01_train_base_eda          (Skylar, EDA only)
02_static_0_preprocessing  (Skylar, EDA only)
        │  findings / rules
        ▼
03_clean_missing_outliers   Issue #3   raw CSV ─► data/interim/applicant_{train,test}.parquet
        ▼
04_feature_engineering      Issue #6   ─► data/processed/features_{train,test}.parquet
        ▼
05_encode_scale             Issue #4   ─► data/processed/model_ready_{fit,valid,test}.parquet
        ▼
models (October)
```
Run 03 → 04 → 05 in order; each notebook reads the file the previous one saved.

**Input:** `data/raw/csv_files/{train,test}/*_base.csv`, `*_static_0_*.csv`
**Output:** `data/interim/applicant_{train,test}.parquet`: one row per applicant, cleaned.

## Why this step comes first

```
01 base EDA ─┐
02 static_0 EDA ─┴─► 03 clean ─► 04 feature engineering ─► 05 encode & scale ─► models
```

Cleaning comes **before** feature engineering and encoding because every later step relies on it:
- **Outliers are capped here, on raw values.** Feature engineering (04) builds ratios like `credit_to_income` from these
  columns, so an extreme income would distort every feature built from it. Capping must happen on the real
  amounts, not on scaled z-scores.
- **The join is checked here.** `base` + `static_0` is validated as exactly one row per `case_id`, which is the table 04 attaches everything else to.
- **Missing values are *not* filled here, on purpose.** Imputation is a *fitted* step (it learns medians), so it happens once, in 05,
  on the final set of columns. Filling here would miss all the new columns 04 creates, which are often missing (e.g. no bureau history).
- The outlier caps are learned from **training weeks < 80 only** and then applied unchanged to validation and test, to avoid leakage.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent                      # notebooks/ -> repo root
RAW = ROOT / "data" / "raw" / "csv_files"     # unzipped dataset (git-ignored, see README)
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

# Train = weeks 0-91, real test = weeks 92-107. Weeks 80-91 are our out-of-time
# validation set, so anything "learned" from data (caps, medians, scalers)
# is fitted on weeks < 80 only.
VALID_START_WEEK = 80

## 1. Load and join base + static_0

Same join as notebook 02: `static_0` comes in two fragments (`_0_0`, `_0_1`) that are stacked, then
joined one-to-one on `case_id`. `MONTH` is dropped because notebook 01 showed it repeats `date_decision`.

In [2]:
def load_applicant(split):
    base = pd.read_csv(RAW / split / f"{split}_base.csv")
    static = pd.concat(
        [pd.read_csv(RAW / split / f"{split}_static_0_{i}.csv") for i in (0, 1)],
        ignore_index=True,
    )
    df = base.merge(static, on="case_id", how="left", validate="one_to_one")
    df["date_decision"] = pd.to_datetime(df["date_decision"])
    df["birth_259D"] = pd.to_datetime(df["birth_259D"])
    return df.drop(columns=["MONTH"])


train = load_applicant("train")
test = load_applicant("test")
print("train", train.shape, "| test", test.shape)
train.head()

train (1000000, 11) | test (200000, 10)


,case_id,date_decision,WEEK_NUM,target,mainoccupationinc_384A,credamount_770A,annuity_780A,days_employed_700P,education_927M,maritalstatus_703M,birth_259D
0,1,2020-04-28,69,0,28314.0,16373.0,600.0,1976.0,6def22f0,f62a4fc7,1969-08-15
1,2,2019-08-13,32,0,25386.0,13707.0,725.0,NaN,10795bac,ea6fb4b5,1952-01-27
2,3,2019-01-01,0,1,57416.0,12676.0,619.0,762.0,242b264b,e78cb9c0,1978-06-07
3,4,2019-08-27,34,1,31763.0,44528.0,830.0,355.0,6def22f0,e78cb9c0,1962-12-24
4,5,2019-03-26,12,0,11990.0,21065.0,1282.0,0.0,6def22f0,f62a4fc7,1997-03-21


## 2. Missing values: keep every row

Is being missing itself related to default? If so, dropping those rows would throw away signal.

In [3]:
num_cols = ["mainoccupationinc_384A", "credamount_770A", "annuity_780A", "days_employed_700P"]

pd.DataFrame({
    "pct_missing": train[num_cols].isna().mean().mul(100),
    "default_rate_if_missing": [train.loc[train[c].isna(), "target"].mean() for c in num_cols],
    "default_rate_if_present": [train.loc[train[c].notna(), "target"].mean() for c in num_cols],
}).round(3)

,pct_missing,default_rate_if_missing,default_rate_if_present
mainoccupationinc_384A,4.965,0.191,0.192
credamount_770A,1.996,0.190,0.192
annuity_780A,9.968,0.194,0.192
days_employed_700P,15.057,0.193,0.191


In [4]:
kept = train.dropna(subset=num_cols)
print(f"Dropping rows with any missing value would remove {1 - len(kept) / len(train):.1%} of applicants.")
print("We also can't drop rows from test: every applicant needs a decision.")

Dropping rows with any missing value would remove 28.8% of applicants.
We also can't drop rows from test: every applicant needs a decision.


**Decision: no rows are dropped.**

What the numbers above show: in `static_0`, missing values look **random**. The default rate is ~19.1-19.4% whether a
value is missing or not, so here missingness is *not* a risk signal. Rows are still kept because:
- dropping would remove **28.8%** of applicants (see above) and shrink the training data for no gain
- test rows can't be dropped, since every applicant needs a decision
- later tables (credit bureau) are missing **entirely** for thin-file applicants, the group this project is about, and
  *that* kind of missingness is strongly linked to default (25.6% vs 16.4%)

Missing values are filled in notebook 05 by Sama's `encode_scale`, fitted on training weeks only:
- numeric columns get the median, plus a `missingindicator_<col>` 0/1 column so the model can still use the fact that a value was missing
- categorical columns get an explicit `"Missing"` category

## 3. Outliers: cap instead of removing

Notebook 02 found right-skewed amounts, with the max around 2.5-3× the 99th percentile. The values are extreme but plausible,
so we **winsorize**: values are clipped to the 1st-99th percentile range.
- The caps are computed on **training weeks < 80 only**, and the same caps are applied to validation and test (no leakage).
- Capping keeps every applicant, and it stops a few huge incomes from dominating `StandardScaler` in notebook 05.

In [5]:
fit_rows = train["WEEK_NUM"] < VALID_START_WEEK
caps = train.loc[fit_rows, num_cols].quantile([0.01, 0.99]).T
caps.columns = ["low", "high"]
caps

,low,high
mainoccupationinc_384A,7521.0,64421.54
credamount_770A,4606.0,47329.06
annuity_780A,342.0,3508.00
days_employed_700P,0.0,3711.00


In [6]:
def cap_outliers(df, caps):
    df = df.copy()
    for col, (low, high) in caps.iterrows():
        df[col] = df[col].clip(low, high)   # NaN stays NaN
    return df


before = train[num_cols].describe().loc[["min", "50%", "max"]]
share_capped = ((train[num_cols] < caps["low"]) | (train[num_cols] > caps["high"])).mean().mul(100).round(2)

train = cap_outliers(train, caps)
test = cap_outliers(test, caps)

after = train[num_cols].describe().loc[["min", "50%", "max"]]
print("% of rows capped per column:\n", share_capped.to_string(), "\n")
pd.concat({"before": before, "after": after})

% of rows capped per column:
 mainoccupationinc_384A    1.90
credamount_770A           1.96
annuity_780A              1.79
days_employed_700P        0.85 



mainoccupationinc_384A  credamount_770A  annuity_780A  \
before min                 2515.00          1447.00          79.0   
       50%                22021.00         14760.00        1097.0   
       max               174473.00        150395.00       12069.0   
after  min                 7521.00          4606.00         342.0   
       50%                22021.00         14760.00        1097.0   
       max                64421.54         47329.06        3508.0   

            days_employed_700P  
before min                 0.0  
       50%              1502.0  
       max              5852.0  
after  min                 0.0  
       50%              1502.0  
       max              3711.0

Sanity check for implausible values: applicant age at decision time.

In [7]:
age = (train["date_decision"] - train["birth_259D"]).dt.days / 365.25
age.describe().round(1)

count    1000000.0
mean          46.3
std           14.1
min           21.0
25%           34.1
50%           46.3
75%           58.5
max           71.7
dtype: float64

## 4. Save for notebook 04

In [8]:
train.to_parquet(INTERIM / "applicant_train.parquet", index=False)
test.to_parquet(INTERIM / "applicant_test.parquet", index=False)
print("saved:", sorted(p.name for p in INTERIM.glob("applicant_*.parquet")))

saved: ['applicant_test.parquet', 'applicant_train.parquet']
